# Selectivity Estimation, and the Errors That Choose the Wrong Plan

Companion notebook for the [Cost Models and Statistics](https://codemia.io/courses/database_internals/dbi_cost_models_and_statistics) lesson on Codemia.

**What you will build**: the statistics a planner actually keeps, which is a histogram and a list of
the most common values, the selectivity estimates it derives from them, and a small cost model that
turns an estimate into a plan choice. By the end you will be able to say which predicates a planner
estimates well, which it estimates catastrophically, and how far wrong an estimate has to be before
the plan changes.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(23)

N = 200000      # rows in the table
PAGES = 2500    # pages the table occupies
BUCKETS = 20    # histogram buckets, close to a real default

## Concept recap

A planner cannot look at the data when it plans. It looks at a summary of the data collected the
last time statistics were gathered, and from that summary it guesses how many rows each operation
will produce. That guess is the selectivity, and almost every plan decision follows from it: whether
to use an index, which join algorithm to use, which input to build a hash table from, how much
memory to request.

The summary is small. For each column it is typically a row count, a count of distinct values, a
list of the most common values with their exact frequencies, and a histogram of everything else,
usually with a hundred or so buckets. That is a few kilobytes standing in for a few gigabytes, so
it is right about some questions and badly wrong about others, and knowing which is which is most
of what reading a plan is.

The error that matters is not the small one. Estimating ten thousand rows instead of twelve thousand
changes nothing. Estimating ten rows instead of a hundred thousand changes the plan, and the usual
cause is not a stale histogram, it is an assumption the planner has no choice but to make.

## A table with the shape real data has

In [ ]:
# amount: heavily skewed, as money columns are
amount = np.round(rng.lognormal(3.2, 1.1, N), 2)

# city and postcode: correlated, because a postcode belongs to a city
CITIES = 60
city = np.minimum(rng.zipf(1.4, N), CITIES) - 1
postcode = city * 3 + rng.integers(0, 3, N)      # each city owns exactly three postcodes

print("rows                %d" % N)
print("distinct amounts    %d" % len(np.unique(amount)))
print("distinct cities     %d" % len(np.unique(city)))
print("distinct postcodes  %d" % len(np.unique(postcode)))
print("most common city    %.1f%% of rows" % (100 * np.bincount(city).max() / N))

## The statistics a planner keeps

In [ ]:
class Stats:
    """An equi depth histogram plus a most common values list, which is what engines store."""

    def __init__(self, values, buckets=BUCKETS, mcv_count=10):
        self.n = len(values)
        vals, counts = np.unique(values, return_counts=True)
        order = np.argsort(counts)[::-1]
        top = order[:mcv_count]
        self.mcv = {float(vals[i]): int(counts[i]) for i in top}
        self.distinct = len(vals)
        rest = values[~np.isin(values, list(self.mcv))]
        self.rest_n = len(rest)
        self.rest_distinct = max(1, self.distinct - len(self.mcv))
        qs = np.linspace(0, 1, buckets + 1)
        self.edges = np.quantile(rest, qs) if len(rest) else np.array([0.0, 0.0])

    def equality(self, v):
        """Rows expected to match `col = v`."""
        if float(v) in self.mcv:
            return self.mcv[float(v)]
        return self.rest_n / self.rest_distinct

    def less_than(self, v):
        """Rows expected to match `col < v`, interpolating inside the containing bucket."""
        per = self.rest_n / (len(self.edges) - 1)
        est = sum(self.mcv[k] for k in self.mcv if k < v)
        for i in range(len(self.edges) - 1):
            lo, hi = self.edges[i], self.edges[i + 1]
            if v >= hi:
                est += per
            elif v > lo:
                est += per * (v - lo) / max(hi - lo, 1e-12)
                break
            else:
                break
        return est

    def between(self, lo, hi):
        return max(0.0, self.less_than(hi) - self.less_than(lo))


amount_stats = Stats(amount)
city_stats = Stats(city, mcv_count=10)
post_stats = Stats(postcode, mcv_count=10)
print("histogram edges kept: %d, most common values kept: %d"
      % (len(amount_stats.edges), len(amount_stats.mcv)))

## What the estimates get right, and the first thing they do not

In [ ]:
def report(rows, label, est, actual):
    ratio = est / actual if actual else float("inf")
    print("%-34s %12.0f %12d %10.1fx" % (label, est, actual, ratio))

print("%-34s %12s %12s %10s" % ("predicate", "estimated", "actual", "error"))
report(N, "amount < 20", amount_stats.less_than(20), int((amount < 20).sum()))
report(N, "amount between 20 and 60", amount_stats.between(20, 60), int(((amount >= 20) & (amount < 60)).sum()))
report(N, "amount > 500", N - amount_stats.less_than(500), int((amount >= 500).sum()))
report(N, "city = 0 (the busiest)", city_stats.equality(0), int((city == 0).sum()))
report(N, "city = 41 (a rare one)", city_stats.equality(41), int((city == 41).sum()))

The first two rows are what a histogram is for. A range in the body of the distribution spans whole
buckets, each bucket holds a known number of rows by construction, and the estimate is correct to
within a per cent. The busiest city is estimated exactly, because its frequency is stored verbatim
in the most common values list rather than derived from anything.

The rare city is the uniformity assumption showing. It is not in the list, so its frequency is the
remaining rows divided by the remaining distinct values, and in a skewed column the values outside
the list are not uniform among themselves either. Being out by a factor of two is the result, and a
longer list removes it.

The fourth row is the one to remember. The predicate `amount > 500` is overestimated by nearly
fifteen times, and the histogram is not stale, badly built or too small. The last bucket of an equi
depth histogram on a skewed column spans an enormous range, from a few hundred to the largest value
in the table, and the estimate inside a bucket is a linear interpolation. The real distribution in
that bucket is nothing like linear, so a predicate that selects the extreme tail gets assigned a
large slice of a bucket that holds almost nothing out there.

Tail predicates on skewed columns are therefore the single estimate most worth distrusting, and they
are extremely common: amounts above a threshold, durations above a limit, counts above a quota. Every
one of them lands in the last bucket.

## What the estimates get catastrophically wrong

In [ ]:
print("%-40s %12s %12s %10s" % ("predicate", "estimated", "actual", "error"))
for c in (0, 2, 7):
    p = c * 3          # a postcode that belongs to city c
    sc = city_stats.equality(c) / N
    sp = post_stats.equality(p) / N
    est = N * sc * sp                      # the independence assumption
    actual = int(((city == c) & (postcode == p)).sum())
    print("%-40s %12.1f %12d %10.3fx"
          % ("city = %d and postcode = %d" % (c, p), est, actual, est / max(actual, 1)))

print()
print("the same two predicates, estimated separately:")
print("  city = 0        estimated %8.0f   actual %8d" % (city_stats.equality(0), (city == 0).sum()))
print("  postcode = 0    estimated %8.0f   actual %8d" % (post_stats.equality(0), (postcode == 0).sum()))

Each predicate on its own is estimated well. Together they are out by two or three orders of
magnitude, and the reason is in one line of the code above: the planner multiplies the two
selectivities, which is only correct if the columns are independent.

They are not independent. A postcode belongs to exactly one city, so every row matching the
postcode already matches the city, and the true selectivity of the pair is the selectivity of the
postcode alone. The planner has multiplied by the city selectivity for nothing.

This is the single most common cause of a badly wrong estimate in production, and it has nothing to
do with stale statistics. Refreshing them changes nothing, because each individual number is
correct. The information that would fix it, that these two columns move together, is not in the
summary at all. That is what multi column or extended statistics exist to record, and why they have
to be asked for explicitly: there are too many column pairs to collect by default.

## How far wrong an estimate has to be to change the plan

In [ ]:
RANDOM_PAGE = 4.0     # cost of fetching one page by index
SEQ_PAGE = 1.0        # cost of fetching one page in a sequential scan
CPU_ROW = 0.01

def seq_cost():
    return PAGES * SEQ_PAGE + N * CPU_ROW

def index_cost(rows):
    return rows * (RANDOM_PAGE + CPU_ROW)

print("sequential scan costs %.0f regardless of the predicate" % seq_cost())
crossover = seq_cost() / (RANDOM_PAGE + CPU_ROW)
print("so the index wins below %.0f rows, which is %.2f%% of the table"
      % (crossover, 100 * crossover / N))
print()
print("%14s %14s %14s %16s" % ("estimated rows", "index cost", "seq cost", "planner chooses"))
for est in (10, 100, 1000, 5000, 20000, 100000):
    print("%14d %14.0f %14.0f %16s"
          % (est, index_cost(est), seq_cost(),
             "index" if index_cost(est) < seq_cost() else "sequential scan"))

In [ ]:
# what the wrong choice actually costs, when the real row count is 60000
true_rows = 60000
estimates = np.array([60, 600, 6000, 60000])
chosen = ["index" if index_cost(e) < seq_cost() else "seq" for e in estimates]
paid = [index_cost(true_rows) if c == "index" else seq_cost() for c in chosen]
best = min(index_cost(true_rows), seq_cost())

print("the predicate really matches %d rows" % true_rows)
print("%14s %10s %14s %12s" % ("estimated", "chooses", "cost paid", "penalty"))
for e, c, p in zip(estimates, chosen, paid):
    print("%14d %10s %14.0f %11.1fx" % (e, c, p, p / best))

Two things worth taking from this.

The first is that an estimate can be wrong by a factor of ten and change nothing. Ten rows or a
hundred, the index wins either way; sixty thousand rows or six hundred thousand, the scan wins
either way. Most estimation error is harmless, which is why planners get away with a few kilobytes
of statistics.

The second is what happens when the error crosses the threshold. An estimate of six hundred rows
against a true sixty thousand picks the index, and the index then performs sixty thousand random
page fetches to produce a result the scan would have produced for a fraction of the cost. The
penalty is not proportional to the estimation error; it is the ratio between two plans that were
never close.

That asymmetry is why the correlated predicate above is dangerous and the rare city is not. One
produces an error of a factor of two in a region where nothing changes. The other produces an error
of a factor of a thousand, straddling the crossover, on a query shape that appears in every
application that stores an address.

## Variations: more buckets, and a longer most common values list

In [ ]:
narrow_a = int(((amount >= 24) & (amount < 26)).sum())
wide_a = int(((amount >= 20) & (amount < 200)).sum())
tail_a = int((amount >= 500).sum())

print("%10s %14s %12s %12s" % ("buckets", "narrow range", "wide range", "the tail"))
for b in (5, 10, 20, 50, 100, 400):
    st = Stats(amount, buckets=b)
    print("%10d %13.2fx %11.2fx %11.2fx"
          % (b, st.between(24, 26) / narrow_a, st.between(20, 200) / wide_a,
             (N - st.less_than(500)) / tail_a))

print()
print("%10s %26s" % ("mcv kept", "error on a rare city"))
for k in (0, 5, 10, 30, 60):
    st = Stats(city, mcv_count=k)
    print("%10d %25.2fx" % (k, st.equality(41) / max((city == 41).sum(), 1)))

The narrow range is accurate at every bucket count, because an equi depth histogram puts its
narrowest buckets where the data is densest and that is where this predicate sits. The wide range
improves from a quarter low to exact. The tail is where the bucket count earns its keep: more
buckets means a narrower last bucket, so less of the table is being described by one linear
interpolation over a long tail.

The most common values list is the more dramatic of the two knobs, because a column with sixty
distinct values and heavy skew can be described exactly once the list is long enough to hold all of
them. That is a real and underused remedy for a skewed low cardinality column, and it costs nothing
at query time.

Neither knob touches the correlation error. It is worth being clear about that, because raising a
statistics target is the reflex when an estimate is wrong, and on the one error in this notebook that
reverses plans by three orders of magnitude it is the wrong reflex.

## What to try next

- Add an extended statistic that stores the joint frequencies of the city and postcode pair, and
  confirm the combined estimate becomes correct. Note how much space it would take for every pair of
  columns in a wide table, which is why it is opt in.
- Estimate a join rather than a filter. The textbook rule divides by the larger distinct count, and
  it degrades in exactly the same way when the join columns are correlated with a filter.
- Make the statistics stale by estimating from the first half of the data and testing against the
  whole, which is what a planner is doing between statistics collections on a growing table.
- Add a second index and a covering index to the cost model and find the region where each wins,
  which is the shape of the decision an engine makes per query.